# DLPFC section 151672

Recompute the dataset-weighted benchmark ARIs in Fig. 2a and the selected
STBSR partition in Fig. 2c. The default mode replays the supplied figure
assignments and benchmark tables. It runs without the raw Visium input.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "config.json").is_file()
             and (p / "STBSR" / "pipeline.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Start Jupyter from the STBSR repository or its notebooks directory.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from STBSR.reproduction import (
    compare_paper_ari, display_labels, evaluate_file, paper_domains, plot_domains,
    read_reference, score_paper_domains,
)

config = json.loads((repo / "config.json").read_text(encoding="utf-8"))
benchmark = repo / "benchmark"
archived_runs = pd.read_csv(benchmark / "stbsr_per_run.csv", dtype={"dataset": str})
mode = os.environ.get("STBSR_RUN_MODE", "paper")  # "paper" or "train"
if mode not in {"paper", "train"}:
    raise ValueError("Use STBSR_RUN_MODE=paper or STBSR_RUN_MODE=train.")
device = os.environ.get("STBSR_DEVICE", "cuda")
data_root = Path(os.environ.get("STBSR_DATA_ROOT", repo / "data"))

## Fig. 2a and Supplementary Data 1

In [ ]:
dataset = "151672"
seed = 11059586  # Recorded seed of the fifth run displayed in Fig. 2c.
spot_size = 4

display(compare_paper_ari().to_frame().round(4))
ari_table = pd.read_csv(benchmark / "mean_ari_by_dataset.csv", dtype={"dataset": str})
ordered = ari_table.drop(columns="dataset").mean().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.boxplot([ari_table[method] for method in ordered], vert=False, tick_labels=ordered,
           showmeans=True, meanprops={"marker": "D", "markerfacecolor": "#355f91"})
ax.set_xlabel("Dataset-level mean ARI")
fig.tight_layout()
display(fig)
plt.close(fig)

dl = ari_table[ari_table.dataset.str.fullmatch(r"151\d{3}")]
display(dl[["STBSR", "GraphST"]].mean().rename("DLPFC mean ARI").to_frame().round(4))
run_scores = archived_runs.loc[archived_runs.dataset.eq(dataset)]
display(run_scores[["run", "seed", "ARI", "NMI", "BF", "BandAcc"]].round(4))
print(f"151672 ARI across ten runs: {run_scores.ARI.mean():.4f} "
      f"(sample SD {run_scores.ARI.std():.4f})")

## Fig. 2c selected STBSR result

In [ ]:
domains = paper_domains(dataset)
ari, n_scored = score_paper_domains(domains, domains["cluster"])
assert n_scored == 3888 and abs(ari - 0.7149832022786045) < 1e-12
fig = plot_domains(domains, {f"STBSR reconstruction (ARI {ari:.4f})": domains["domain"]}, size=spot_size)
display(fig)
plt.close(fig)

## Fig. 2e and f measured markers and reconstructed representation

In [ ]:
display(pd.read_csv(benchmark / "151672_fixed_marker_statistics.csv"))
display(pd.read_csv(benchmark / "151672_PAGA.csv", index_col=0).round(3))

with np.load(benchmark / "151672_downstream_arrays.npz", allow_pickle=False) as saved:
    row_index = pd.Index(saved["ids"])
    positions = domains.set_index("spot_id").loc[row_index, ["x", "y"]].to_numpy()
    genes, expression = saved["genes"], saved["expression"]
    umap, dpt = saved["umap"], saved["pseudotime"]
    category = pd.Categorical(saved["display_cluster"]).codes
fig, axes = plt.subplots(1, len(genes), figsize=(12, 3))
for i, ax in enumerate(axes):
    scores = expression[:, i]
    ax.scatter(*positions.T, c=scores, s=4, cmap="viridis", vmax=np.quantile(scores, .99))
    ax.set_title(genes[i])
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_axis_off()
fig.tight_layout()
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].scatter(*umap.T, c=category, s=4, cmap="tab10")
axes[0].set_title("Reconstructed-expression UMAP")
sc = axes[1].scatter(*positions.T, c=dpt, s=4, cmap="viridis")
axes[1].set_title("Diffusion ordering")
axes[1].set_aspect("equal")
axes[1].invert_yaxis()
for ax in axes:
    ax.set_axis_off()
fig.colorbar(sc, ax=axes[1], shrink=.7)
display(fig)
plt.close(fig)

## Optional retraining

Put the original section in `<data-root>/1.DLPFC/151672` with the raw
Visium matrix, tissue positions and `151672_truth.txt`. Set
`STBSR_DATA_ROOT` and `STBSR_RUN_MODE=train`. The displayed run
originally trained on CUDA with a CPU endpoint replay before KMeans.
Use the stored assignments above for exact panel reproduction.

In [ ]:
if mode == "train":
    spec = config["dataset_specs"][dataset]
    data_path = data_root / spec["path"]
    if not data_path.exists():
        raise FileNotFoundError(f"Set STBSR_DATA_ROOT. Missing: {data_path}")

    outputs = repo / "outputs"
    outputs.mkdir(exist_ok=True)
    out = Path(mkdtemp(prefix=f"{dataset}_seed{seed}_", dir=outputs))
    replay = config.get("benchmark_replay_devices", {}).get(dataset, device)
    if device == "cpu":
        replay = "cpu"
    command = [
        sys.executable, "-m", "STBSR.cli", "train",
        "--dataset", dataset, "--input", str(data_path), "--seed", str(seed),
        "--device", device, "--replay-device", replay, "--out", str(out),
    ]
    subprocess.run(command, cwd=repo, check=True, timeout=config["maximum_case_seconds"] + 180)

    reference = read_reference(dataset, spec, data_path)
    prediction_path = out / "predictions.npz"
    results = pd.DataFrame(evaluate_file(prediction_path, reference)).set_index("readout")
    display(results[["n_scored", "ARI", "NMI", "BF", "BandAcc"]])

    with np.load(prediction_path, allow_pickle=False) as values:
        coords = values["coordinates"]
        frame = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1]})
        frame["reference"] = reference.reindex(values["barcodes"]).to_numpy()
        replotted = {
            f"{label} (ARI {results.loc[key, 'ARI']:.4f})":
                display_labels(frame["reference"], pd.Series(values[key], index=frame.index))
            for key, label in (("embedding64", "64D"), ("reconstruction_unit", "Reconstruction"))
        }
    fig = plot_domains(frame, replotted, size=spot_size)
    display(fig)
    plt.close(fig)
    print("Training output:", out)
else:
    print("Paper replay completed. Enable training with STBSR_RUN_MODE=train.")